# Predictive Performance

**Fine-tuned EmbeddingGemma 2 version** of `main/code/03_1_predictive_performance_txt_img.ipynb`: the DoubleMLDeep product embeddings are replaced by [EmbeddingGemma 2](https://huggingface.co/google/embeddinggemma-2) embeddings fine-tuned with LoRA and the loss of the DoubleMLDeep model on the training sample (`finetune_gemma.py`, `00_finetune_gemma_colab.ipynb`; text + image, Matryoshka 256 dimensions). Datasets are read from `Gemma-tuned/data`, raw data from `main/data`, outputs are written to `Gemma-tuned/outputs`.

In [1]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)

/tmp/claude-0/-home-user-demand-analysis-repro/8871aba3-e06d-5440-87b8-bf4072e29cf1/scratchpad/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
txt_only = False
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"
df_full_train = pd.read_csv(f"../data/{dataframe_name}_train.zip")
df_full_val = pd.read_csv(f"../data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2",
    "P_bb_t-2",
    "REVIEW_COUNT_t-2",
    "RATING_t-2",
    # 'pred_ml_l_diff_lag_2',
    # 'pred_ml_m_diff_lag_2',
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val = df_full_val.dropna()

dummy_subcat_names = [
    category for category in df_full_val["subcat_aggregated"].unique()
]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Dummy subcat names: ['Residual', 'Cars & Race Cars', 'Trains & Trams', 'Train Sets', 'Trucks', 'Tractors', 'Race Tracks', 'Airplanes', 'Motor Vehicles', 'Vehicle Playsets', 'Skateboards']
All time steps: ['2023-02-27', '2023-03-27', '2023-04-24', '2023-05-22', '2023-06-19', '2023-07-17', '2023-08-14', '2023-09-11', '2023-10-09', '2023-11-06', '2023-12-04', '2024-01-01', '2024-01-29']


Index(['ASIN', 'date', 'index', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=344)

In [3]:
print(len(df_full_val))

38041


In [4]:
print(len(df_full_train))

38146


## Variable Names

In [5]:
n_lags = 1

outcome = "Q_t"
treatment = "P_bb_t"

outcome_diff = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + [
        "Lightning Deals: Upcoming Deal",
        "Buy Box: Is FBA",
    ]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = [
    "RATING",
    "REVIEW_COUNT",
]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0",
    "similarity_cluster_1",
    "similarity_cluster_2",
    "similarity_cluster_3",
    "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls

additional_controls_deep = [
    var for var in additional_controls if var not in dummy_subcat_names
]

controls_emb = [var for var in df_full_train.columns if "emb" in var]

In [6]:
# dataframe for all results
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)

## Performance Deep Model with time independent controls

The deep models are the DoubleMLDeep models of the main analysis (`pred_ml_*` are taken unchanged from the datasets of the main analysis). They are reported as benchmark for the predictive performance of the fine-tuned EmbeddingGemma 2 features below. The row `Fine-tuned EmbeddingGemma 2 (heads)` reports the predictions of the two heads trained together with the LoRA adapters (`finetune_gemma.py`, `data/finetuned/txtimg/predictions_txtimg.parquet`; time-independent, levels only), the counterpart of `Deep Time Independent`.

In [7]:
df_dict = {
    "Train": df_full_train,
    "Test": df_full_val,
}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome: {r2_ml_l}")
    print(f"R2 Treatment: {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome (time dependent): {r2_ml_l_lag1}")
    print(f"R2 Treatment (time dependent): {r2_ml_m_lag1}")

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

# predictions of the heads of the fine-tuned EmbeddingGemma 2 model (time-independent, levels only)
pred_tuned = pd.read_parquet("../data/finetuned/txtimg/predictions_txtimg.parquet")
pred_tuned["date"] = pred_tuned["date"].astype(str)
for df_name, df in df_dict.items():
    p = df[["ASIN", "date"]].astype({"date": str}).merge(pred_tuned, on=["ASIN", "date"], how="left", validate="one_to_one")
    assert p[["pred_ml_l", "pred_ml_m"]].notna().all().all()
    results_df_deep.loc["Fine-tuned EmbeddingGemma 2 (heads)", f"R2 Q {df_name}"] = np.round(r2_score(df[outcome].values, p["pred_ml_l"].values), 4)
    results_df_deep.loc["Fine-tuned EmbeddingGemma 2 (heads)", f"R2 P {df_name}"] = np.round(r2_score(df[treatment].values, p["pred_ml_m"].values), 4)

results_df_deep

Evaluation for Train set
R2 Outcome: 0.852
R2 Treatment: 0.9601
Evaluation for Train set
R2 Outcome (time dependent): 0.8576
R2 Treatment (time dependent): 0.9569
Evaluation for Test set
R2 Outcome: 0.5044
R2 Treatment: 0.6726
Evaluation for Test set
R2 Outcome (time dependent): 0.5953
R2 Treatment (time dependent): 0.6669


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.8520,0.5044,0.9601,0.6726
Deep Time Dependent,0.8576,0.5953,0.9569,0.6669
Fine-tuned EmbeddingGemma 2 (heads),0.8653,0.5000,0.9692,0.7419


In [8]:
df_dict = {
    "Train": df_full_train,
    "Test": df_full_val,
}

results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome: {r2_ml_l_diff}")
    print(f"R2 Treatment: {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"Evaluation for {df_name} set")
    print(f"R2 Outcome (time dependent): {r2_ml_l_diff_lag1}")
    print(f"R2 Treatment (time dependent): {r2_ml_m_diff_lag1}")

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

Evaluation for Train set
R2 Outcome: 0.28147266
R2 Treatment: 0.08475598
Evaluation for Train set
R2 Outcome (time dependent): 0.42506891
R2 Treatment (time dependent): 0.24821505
Evaluation for Test set
R2 Outcome: 0.10888856
R2 Treatment: 0.01153205
Evaluation for Test set
R2 Outcome (time dependent): 0.12168179
R2 Treatment (time dependent): 0.0165301


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.281473,0.108889,0.084756,0.011532
Deep Time Dependent,0.425069,0.121682,0.248215,0.016530


## Feature generation

In [9]:
# train set
y_train = df_full_train[outcome].squeeze()
d_train = df_full_train[treatment].squeeze()
y_train_diff = df_full_train[outcome_diff].squeeze()
d_train_diff = df_full_train[treatment_diff].squeeze()

x_train = df_full_train[additional_controls]
x_train = sm.add_constant(x_train)

x_train_pca = df_full_train[additional_controls + controls_pca]
x_train_pca = sm.add_constant(x_train_pca)
x_train_sim = df_full_train[additional_controls + controls_similarities]
x_train_sim = sm.add_constant(x_train_sim)
x_train_emb = df_full_train[additional_controls + controls_emb]
x_train_emb = sm.add_constant(x_train_emb)

# test set
y_test = df_full_val[outcome].squeeze()
d_test = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test = df_full_val[additional_controls]
x_test = sm.add_constant(x_test)

x_test_pca = df_full_val[additional_controls + controls_pca]
x_test_pca = sm.add_constant(x_test_pca)
x_test_sim = df_full_val[additional_controls + controls_similarities]
x_test_sim = sm.add_constant(x_test_sim)
x_test_emb = df_full_val[additional_controls + controls_emb]
x_test_emb = sm.add_constant(x_test_emb)

# update names for boosting
x_train.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_pca.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_pca.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_sim.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_sim.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

x_train_emb.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)
x_test_emb.rename(columns=lambda x: re.sub("[^A-Za-z0-9_]+", "", x), inplace=True)

In [10]:
train_dict = {
    "y": y_train,
    "y_diff": y_train_diff,
    "d": d_train,
    "d_diff": d_train_diff,
    "x": x_train,
    "x_emb": x_train_emb,
    "x_pca": x_train_pca,
    "x_sim": x_train_sim,
}

test_dict = {
    "y": y_test,
    "y_diff": y_test_diff,
    "d": d_test,
    "d_diff": d_test_diff,
    "x": x_test,
    "x_emb": x_test_emb,
    "x_pca": x_test_pca,
    "x_sim": x_test_sim,
}

## Performance Level Data

In [11]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print("\n ------------------------------------")
    print(f"Feature specification: {feature_specification}")

    print("\nOLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train = r2_score(
        train_dict["y"], outcome_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Outcome (train): {ols_outcome_train}")
    ols_outcome_test = r2_score(
        test_dict["y"], outcome_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Outcome (test): {ols_outcome_test}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(
        train_dict["d"], treatment_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Treatment (train): {ols_treatment_train}")
    ols_treatment_test = r2_score(
        test_dict["d"], treatment_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Treatment (test): {ols_treatment_test}")

    print("\nBoosting")
    boost_q_outcome = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_outcome.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q_outcome.score(
        train_dict[feature_specification], train_dict["y"]
    )
    print(f"R2 Outcome (train): {r2_boost_q_train}")
    r2_boost_q_test = boost_q_outcome.score(
        test_dict[feature_specification], test_dict["y"]
    )
    print(f"R2 Outcome (test): {r2_boost_q_test}")

    boost_q_treatment = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_treatment.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_q_treatment.score(
        train_dict[feature_specification], train_dict["d"]
    )
    print(f"R2 Treatment (train): {r2_boost_d_train}")
    r2_boost_d_test = boost_q_treatment.score(
        test_dict[feature_specification], test_dict["d"]
    )
    print(f"R2 Treatment (test): {r2_boost_d_test}")

    results_df_tab_tmp = pd.DataFrame(
        data=[
            [
                ols_outcome_train,
                ols_outcome_test,
                ols_treatment_train,
                ols_treatment_test,
            ],
            [r2_boost_q_train, r2_boost_q_test, r2_boost_d_train, r2_boost_d_test],
        ],
    )
    results_df_tab = pd.concat([results_df_tab, results_df_tab_tmp], axis=0)

row_names = [
    "OLS (Tabular)",
    "Boosting (Tabular)",
    "OLS (Tabular + PCA)",
    "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)",
    "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)",
    "Boosting (Tabular + Embeddings)",
]

results_df_tab.columns = column_names
results_df_tab.index = row_names

results_df_tab


 ------------------------------------
Feature specification: x

OLS
R2 Outcome (train): 0.21370864863876404
R2 Outcome (test): 0.20838958759549508


R2 Treatment (train): 0.16152708611454358
R2 Treatment (test): 0.1581512127497423

Boosting


R2 Outcome (train): 0.760628763684153
R2 Outcome (test): 0.4750710061808484


R2 Treatment (train): 0.7369382567656562
R2 Treatment (test): 0.18535617147478323

 ------------------------------------
Feature specification: x_pca

OLS
R2 Outcome (train): 0.8074119692977083
R2 Outcome (test): 0.48164878624627117


R2 Treatment (train): 0.898805911123175
R2 Treatment (test): 0.7031046867286984

Boosting


...

R2 Outcome (train): 0.9449536957574255


R2 Outcome (test): 0.5557373196639885


R2 Treatment (train): 0.9835385243837271


R2 Treatment (test): 0.7161729335309451


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.213709,0.208390,0.161527,0.158151
Boosting (Tabular),0.760629,0.475071,0.736938,0.185356
OLS (Tabular + PCA),0.807412,0.481649,0.898806,0.703105
Boosting (Tabular + PCA),0.923837,0.544223,0.978188,0.726722
OLS (Tabular + Similarities),0.388906,0.272991,0.896270,0.694258
Boosting (Tabular + Similarities),0.899565,0.520513,0.977588,0.717835
OLS (Tabular + Embeddings),0.862450,0.521952,0.946689,0.726084
Boosting (Tabular + Embeddings),0.944954,0.555737,0.983539,0.716173


Summary

In [12]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.213709,0.208390,0.161527,0.158151
Boosting (Tabular),0.760629,0.475071,0.736938,0.185356
OLS (Tabular + PCA),0.807412,0.481649,0.898806,0.703105
Boosting (Tabular + PCA),0.923837,0.544223,0.978188,0.726722
OLS (Tabular + Similarities),0.388906,0.272991,0.896270,0.694258
Boosting (Tabular + Similarities),0.899565,0.520513,0.977588,0.717835
OLS (Tabular + Embeddings),0.862450,0.521952,0.946689,0.726084
Boosting (Tabular + Embeddings),0.944954,0.555737,0.983539,0.716173
Deep Time Independent,0.852000,0.504400,0.960100,0.672600
Deep Time Dependent,0.857600,0.595300,0.956900,0.666900


## Performance Diff Data

In [13]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print("\n ------------------------------------")
    print(f"Feature specification: {feature_specification}")

    print("\nOLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train = r2_score(
        train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Outcome (train): {ols_outcome_train}")
    ols_outcome_test = r2_score(
        test_dict["y_diff"], outcome_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Outcome (test): {ols_outcome_test}")

    treatment_reg = sm.OLS(
        train_dict["d_diff"], train_dict[feature_specification]
    ).fit()
    ols_treatment_train = r2_score(
        train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification])
    )
    print(f"R2 Treatment (train): {ols_treatment_train}")
    ols_treatment_test = r2_score(
        test_dict["d_diff"], treatment_reg.predict(test_dict[feature_specification])
    )
    print(f"R2 Treatment (test): {ols_treatment_test}")

    print("\nBoosting")
    boost_q_outcome = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_outcome.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q_outcome.score(
        train_dict[feature_specification], train_dict["y_diff"]
    )
    print(f"R2 Outcome (train): {r2_boost_q_train}")
    r2_boost_q_test = boost_q_outcome.score(
        test_dict[feature_specification], test_dict["y_diff"]
    )
    print(f"R2 Outcome (test): {r2_boost_q_test}")

    boost_q_treatment = LGBMRegressor(
        n_estimators=500, lr=0.02, random_state=42, verbose=-1
    )
    boost_q_treatment.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_q_treatment.score(
        train_dict[feature_specification], train_dict["d_diff"]
    )
    print(f"R2 Treatment (train): {r2_boost_d_train}")
    r2_boost_d_test = boost_q_treatment.score(
        test_dict[feature_specification], test_dict["d_diff"]
    )
    print(f"R2 Treatment (test): {r2_boost_d_test}")

    results_df_diff_tab_tmp = pd.DataFrame(
        data=[
            [
                ols_outcome_train,
                ols_outcome_test,
                ols_treatment_train,
                ols_treatment_test,
            ],
            [r2_boost_q_train, r2_boost_q_test, r2_boost_d_train, r2_boost_d_test],
        ],
    )
    results_df_diff_tab = pd.concat(
        [results_df_diff_tab, results_df_diff_tab_tmp], axis=0
    )

row_names = [
    "OLS (Tabular)",
    "Boosting (Tabular)",
    "OLS (Tabular + PCA)",
    "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)",
    "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)",
    "Boosting (Tabular + Embeddings)",
]

results_df_diff_tab.columns = column_names
results_df_diff_tab.index = row_names

results_df_diff_tab


 ------------------------------------
Feature specification: x

OLS
R2 Outcome (train): 0.07640248387937332
R2 Outcome (test): 0.07077318277316202
R2 Treatment (train): 0.005791066894051733


R2 Treatment (test): 0.006365797810488738

Boosting


R2 Outcome (train): 0.3513182806246494
R2 Outcome (test): 0.1513066591847848


R2 Treatment (train): 0.23568023994092024
R2 Treatment (test): -0.004267409128240862

 ------------------------------------
Feature specification: x_pca

OLS


R2 Outcome (train): 0.0764959472129344
R2 Outcome (test): 0.07076365456951372
R2 Treatment (train): 0.006308343496240432
R2 Treatment (test): 0.0066877887287367654

Boosting


...

R2 Outcome (train): 0.08334402005627983
R2 Outcome (test): 0.07224885049053453


R2 Treatment (train): 0.012700011810878098
R2 Treatment (test): 0.004641578858593398

Boosting


R2 Outcome (train): 0.5900396821173923
R2 Outcome (test): 0.18391878930808725


R2 Treatment (train): 0.4342431110282984
R2 Treatment (test): 0.018856277805224475


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.076402,0.070773,0.005791,0.006366
Boosting (Tabular),0.351318,0.151307,0.235680,-0.004267
OLS (Tabular + PCA),0.076496,0.070764,0.006308,0.006688
Boosting (Tabular + PCA),0.468478,0.165540,0.326555,0.007933
OLS (Tabular + Similarities),0.076504,0.070784,0.006218,0.006630
Boosting (Tabular + Similarities),0.447389,0.171867,0.317864,0.004122
OLS (Tabular + Embeddings),0.083344,0.072249,0.012700,0.004642
Boosting (Tabular + Embeddings),0.590040,0.183919,0.434243,0.018856


Summary

In [14]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.076402,0.070773,0.005791,0.006366
Boosting (Tabular),0.351318,0.151307,0.235680,-0.004267
OLS (Tabular + PCA),0.076496,0.070764,0.006308,0.006688
Boosting (Tabular + PCA),0.468478,0.165540,0.326555,0.007933
OLS (Tabular + Similarities),0.076504,0.070784,0.006218,0.006630
Boosting (Tabular + Similarities),0.447389,0.171867,0.317864,0.004122
OLS (Tabular + Embeddings),0.083344,0.072249,0.012700,0.004642
Boosting (Tabular + Embeddings),0.590040,0.183919,0.434243,0.018856
Deep Time Independent,0.281473,0.108889,0.084756,0.011532
Deep Time Dependent,0.425069,0.121682,0.248215,0.016530


## Summary

In [15]:
results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100

,R2 Q Test,R2 P Test
OLS (Tabular),20.84,15.82
Boosting (Tabular),47.51,18.54
OLS (Tabular + PCA),48.16,70.31
Boosting (Tabular + PCA),54.42,72.67
OLS (Tabular + Similarities),27.30,69.43
Boosting (Tabular + Similarities),52.05,71.78
OLS (Tabular + Embeddings),52.20,72.61
Boosting (Tabular + Embeddings),55.57,71.62
Deep Time Independent,50.44,67.26
Deep Time Dependent,59.53,66.69


In [16]:
results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100

,R2 Q Test,R2 P Test
OLS (Tabular),7.08,0.64
Boosting (Tabular),15.13,-0.43
OLS (Tabular + PCA),7.08,0.67
Boosting (Tabular + PCA),16.55,0.79
OLS (Tabular + Similarities),7.08,0.66
Boosting (Tabular + Similarities),17.19,0.41
OLS (Tabular + Embeddings),7.22,0.46
Boosting (Tabular + Embeddings),18.39,1.89
Deep Time Independent,10.89,1.15
Deep Time Dependent,12.17,1.65
